# Employee Attrition Analysis & Prediction

**Dataset:** IBM HR Analytics Employee Attrition Dataset (1,470 employees, 35 columns)
**Goal:** Find the drivers of attrition, predict at-risk employees, and produce outputs for an HR-facing Power BI dashboard.

This notebook covers:
1. Data Cleaning
2. Exploratory Data Analysis (EDA)
3. Modeling (Logistic Regression, Random Forest, XGBoost)
4. Feature Importance ("why" story for HR)
5. Risk Scoring + export for Power BI


## Setup

Import the libraries needed for data handling (`pandas`, `numpy`), visualization (`matplotlib`, `seaborn`), preprocessing (`OrdinalEncoder`, `StandardScaler`), and the three models being compared (`LogisticRegression`, `RandomForestClassifier`, `XGBClassifier`), plus evaluation metrics suited to an imbalanced classification problem.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
from xgboost import XGBClassifier

## Load the data

Read in the raw IBM HR Analytics CSV.

In [ ]:
employees_df = pd.read_csv('IBM HR Analytics Dataset.csv')

## Phase 1 — Data Cleaning

First, get a general health check on the dataset: shape, missing values, duplicates, data types, and how many unique values each column holds. This tells us what needs fixing before anything else.

In [ ]:
print("Initial DataFrame shape:", employees_df.shape)
print()
print("Missing values in each column:\n", employees_df.isnull().sum())
print()
print("Duplicate rows in the DataFrame:", employees_df.duplicated().sum())
print()
print("Data types of each column:\n", employees_df.dtypes)
print()
print("Unique values in each column:\n", employees_df.nunique())
print()

### Drop constant columns and map the target

`EmployeeCount`, `Over18`, and `StandardHours` have the same value in every row (confirmed via `nunique()` above) — they carry no predictive signal, so they're dropped.

`Attrition` and `OverTime` are both binary Yes/No text fields. They're mapped to 1/0 here — early, before encoding — so that every step downstream (EDA groupbys, correlation, modeling) works with clean numeric values instead of text.

In [ ]:
# Dropping constant columns
employees_df.drop(['EmployeeCount', 'Over18', 'StandardHours'], axis=1, inplace=True)
employees_df[['Attrition', 'OverTime']] = employees_df[['Attrition', 'OverTime']].apply(lambda col: col.map({'Yes': 1, 'No': 0}))

### Encode nominal categorical columns

Columns with no inherent order — `BusinessTravel`, `Department`, `EducationField`, `Gender`, `JobRole`, `MaritalStatus` — are one-hot encoded with `drop_first=True` to avoid redundant columns. This produces `encoded_df`, the model-ready version of the data, while `employees_df` is kept in its original, human-readable form for EDA and for the final Power BI export.

Ranked columns already present in the dataset as clean integers (e.g. `JobSatisfaction`, `EnvironmentSatisfaction`, `WorkLifeBalance`, `Education`) are left as-is — they're already ordinal, no encoding needed.

In [ ]:
categorical_cols = [
    'BusinessTravel',
    'Department',
    'EducationField',
    'Gender',
    'JobRole',
    'MaritalStatus'
]
encoded_df = pd.get_dummies(employees_df, columns=categorical_cols, drop_first=True)

### Create derived fields

Three new fields, each capturing something the raw columns don't show on their own:

- **`TenureBucket`** — groups `YearsAtCompany` into readable ranges (0-2, 3-5, 6-10, 11-20, 21-40), useful for readable charts and for testing whether attrition risk clusters at particular tenure stages.
- **`AvgIncomeAtLevel`** — the average `MonthlyIncome` for each `JobLevel`, computed with `.transform('mean')` so the group average is broadcast back onto every row at that level.
- **`IncomeVsLevelAvg`** — each employee's income as a ratio to their level's average. A value below 1.0 flags someone paid below their peers at the same level — a more targeted signal than raw income alone.

In [ ]:
encoded_df['TenureBucket'] = pd.cut(employees_df['YearsAtCompany'], bins=[-1, 2, 5, 10, 20, 40], labels=['0-2', '3-5', '6-10', '11-20', '21-40'])
encoded_df['AvgIncomeAtLevel'] = encoded_df.groupby('JobLevel')['MonthlyIncome'].transform('mean')
encoded_df['IncomeVsLevelAvg'] = encoded_df['MonthlyIncome'] / encoded_df['AvgIncomeAtLevel']

## Phase 2 — Exploratory Data Analysis

EDA is done on `employees_df` (the readable version), before the categorical columns were split apart by one-hot encoding — so `groupby('Department')`, `groupby('JobRole')`, etc. still work cleanly with their original category names.

### Age buckets

`Age` is grouped into bands the same way tenure was, so attrition rate can be compared across life stages instead of dozens of individual ages.

In [ ]:
employees_df['AgeBucket'] = pd.cut(employees_df['Age'], bins=[17,25,35,45,60], labels=['18-25','26-35','36-45','46-60'])

### Correlation heatmap

Checks how strongly the numeric satisfaction, income, and tenure columns move together with `Attrition`. This only captures **linear** relationships — it's a first-pass signal, not the final word on drivers (that comes from the models' feature importances later).

In [ ]:
cols = ['JobSatisfaction', 'EnvironmentSatisfaction', 'WorkLifeBalance',
        'MonthlyIncome', 'YearsAtCompany', 'TotalWorkingYears', 'Attrition', 'OverTime']
corr = employees_df[cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='viridis')
plt.tight_layout()
plt.show()

### Attrition rate by category

A 3x2 grid comparing attrition rate across Department, JobRole, OverTime, MaritalStatus, and AgeBucket — one bar chart per category, looped rather than repeated manually. The 6th (unused) subplot slot is removed with `fig.delaxes()` since there are only 5 categories.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 10))
group_cols = ['Department', 'JobRole', 'OverTime', 'MaritalStatus', 'AgeBucket']

for ax, col in zip(axes.flat, group_cols):
    data = employees_df.groupby(col)['Attrition'].mean()
    sns.barplot(x=data.index, y=data.values, ax=ax, palette='Blues')
    ax.set_xlabel('')
    ax.set_title(f'Attrition by {col}')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
fig.delaxes(axes[2,1])
plt.show()

## Phase 3 — Modeling

### Encode TenureBucket for modeling

`TenureBucket` was created as text labels (`'0-2'`, `'3-5'`, ...) for readability during EDA. Before modeling, it's converted to ordinal numbers — it IS naturally ordered, so `OrdinalEncoder` (not one-hot) is the right fit, preserving the 0-2 < 3-5 < 6-10 ... ranking.

In [ ]:
tenure_order = [['0-2', '3-5', '6-10', '11-20', '21-40']]
encoded_df['TenureBucket'] = OrdinalEncoder(categories=tenure_order).fit_transform(encoded_df[['TenureBucket']])

### Train/test split

`Attrition` is separated out as the target (`target`); everything else in `encoded_df` becomes the feature set (`data`). An 80/20 split is used, with `random_state=42` for reproducibility.

In [ ]:
data = encoded_df.drop(['Attrition'], axis=1)
target = encoded_df['Attrition']
x_train, x_test, y_train, y_test = train_test_split(data, target, test_size=0.2, random_state=42)

### Scale features (Logistic Regression only)

Logistic Regression relies on gradient descent and a regularization penalty, both of which are distorted when features sit on very different scales (e.g. `MonthlyIncome` in the thousands vs. a 0/1 flag). `StandardScaler` is fit on the training set and applied to both train and test sets.

Random Forest and XGBoost split on thresholds, not distances, so they train on the unscaled `x_train`/`x_test` directly — no scaling needed for those two.

In [ ]:
scaler = StandardScaler()
scaled_xtrain = scaler.fit_transform(x_train)
scaled_xtest = scaler.transform(x_test)

### Handle class imbalance and train all three models

Only ~16% of employees in this dataset actually left, so without correcting for imbalance, a model can score well on accuracy while barely detecting anyone who's actually at risk.

- **Logistic Regression** and **Random Forest** both support `class_weight='balanced'`, which up-weights the minority (Attrition=1) class during training.
- **XGBoost** doesn't have `class_weight` — instead it uses `scale_pos_weight`, set here to the ratio of the majority to minority class counts in the training data.

In [ ]:
logistic_regression = LogisticRegression(class_weight='balanced')
random_forest = RandomForestClassifier(class_weight='balanced')
scale = (y_train == 0).sum() / (y_train == 1).sum()
xgboost_classifier = XGBClassifier(scale_pos_weight=scale)

logistic_train = logistic_regression.fit(scaled_xtrain, y_train)
forest_train = random_forest.fit(x_train, y_train)
xgboost_train = xgboost_classifier.fit(x_train, y_train)

### Predict on the test set

Logistic Regression predicts on the **scaled** test set; the two tree-based models predict on the unscaled version, matching how each was trained.

In [ ]:
logistic_predict = logistic_train.predict(scaled_xtest)
forest_predict = forest_train.predict(x_test)
xgboost_predict = xgboost_train.predict(x_test)

### Evaluate all three models

Accuracy alone is misleading on this imbalanced data — a model that just predicts "stays" for everyone would still score ~84% accuracy while catching zero actual leavers. **Precision, Recall, and F1** are computed alongside Accuracy for a fuller picture; **Recall** in particular matters most here, since missing an employee who's about to leave is the costlier mistake for HR than a false alarm.

In [ ]:
logistic_accuracy = accuracy_score(y_test, logistic_predict)
forest_accuracy = accuracy_score(y_test, forest_predict)
xgboost_accuracy = accuracy_score(y_test, xgboost_predict)

logistic_f1 = f1_score(y_test, logistic_predict)
forest_f1 = f1_score(y_test, forest_predict)
xgboost_f1 = f1_score(y_test, xgboost_predict)

logistic_recall = recall_score(y_test, logistic_predict)
forest_recall = recall_score(y_test, forest_predict)
xgboost_recall = recall_score(y_test, xgboost_predict)

logistic_precision = precision_score(y_test, logistic_predict)
forest_precision = precision_score(y_test, forest_predict)
xgboost_precision = precision_score(y_test, xgboost_predict)

results = pd.DataFrame({
    'Model': ['Logistic', 'Random Forest', 'XGBoost'],
    'Accuracy': [logistic_accuracy, forest_accuracy, xgboost_accuracy],
    'Precision': [logistic_precision, forest_precision, xgboost_precision],
    'Recall': [logistic_recall, forest_recall, xgboost_recall],
    'F1': [logistic_f1, forest_f1, xgboost_f1]
})
print(f'Score Comparison Across Various Metrices:\n{results}')
print()

**Result:** Random Forest posts the highest accuracy but the lowest Recall — it plays it safe by leaning toward the majority class. Logistic Regression trades some accuracy for the best Recall by far, meaning it catches the most actual leavers. For this HR use case, **Logistic Regression is the model carried forward** for feature importance and risk scoring.

## Feature Importances — the "why" story for HR

Each model can report which features it relied on most:

- **Logistic Regression** → `.coef_` (signed — positive means the feature increases attrition likelihood, negative means it decreases it)
- **Random Forest** / **XGBoost** → `.feature_importances_` (unsigned strength only)

All three are combined into a single table for comparison. `AbsImportance` (absolute value of the Logistic coefficients) is added and used to sort the table, since Logistic coefficients can be negative — sorting by raw value would bury strong negative predictors at the bottom.

In [ ]:
logistic_imp = logistic_train.coef_[0]
forest_imp = forest_train.feature_importances_
xgboost_imp = xgboost_train.feature_importances_

importance_df = pd.DataFrame({
    'Features': data.columns,
    'Logistic': logistic_imp,
    'Random Forest': forest_imp,
    'XGBoost': xgboost_imp
})

importance_df['AbsImportance'] = importance_df['Logistic'].abs()

importance_df = importance_df.sort_values('AbsImportance', ascending=False)
print(f'Sorted Feature Importances For Every Model:\n{importance_df}')

## Risk scoring and export for Power BI

Rather than scoring only the 20% test split, the trained Logistic Regression model is used to generate a **risk probability for every employee** in the full dataset. `predict_proba(...)[:, 1]` returns the probability of class 1 (Attrition = Yes) for each row; this is converted to a 0–100 scale rounded to 1 decimal place so it reads as a percentage-style risk score.

The score is attached directly onto `employees_df` — the readable version of the data, with `Department`, `JobRole`, etc. still as plain text — since that's the version Power BI will import for the dashboard's Risk page. This only works safely because no rows were reordered or filtered between `employees_df` and `data`/`scaled_full` — row order stayed aligned throughout.

Two files are exported:
- `Updated Employees Data.csv` — full employee data + RiskScore, for the Power BI dashboard
- `Feature Importances.csv` — the sorted importance table, for the Drivers page and the one-pager summary

In [ ]:
scaled_full = scaler.transform(data)
all_risk_scores = logistic_train.predict_proba(scaled_full)[:, 1]
employees_df['RiskScore'] = (all_risk_scores * 100).round(1)

employees_df.to_csv('Updated Employees Data.csv', index=False)
importance_df.to_csv('Feature Importances.csv', index=False)

## Next steps

These two CSVs feed directly into the Power BI dashboard (Overview, Drivers, Risk pages). See the project's README for the top attrition drivers and recommendations distilled from this analysis.